# Lab 3 · Score with the model — registry & real-time endpoint

Two ways to score sample residents: **(A)** load the registered model and score in-notebook, and **(B)** call the **real-time REST endpoint** (Preview) for a live prediction. The sample rows come from `gold.resident_360`; there is no fabricated Rahim row.

> **Attach** the `lh_resident360` Lakehouse first. Run Lab 3 notebook `08` first.

## A. Score from the model registry (no endpoint required)

In [ ]:
import mlflow, pandas as pd
REGISTERED = 'resident360_disengagement'
model = mlflow.pyfunc.load_model(f'models:/{REGISTERED}/latest')

risk_map = {'Not Screened': 0, 'Low': 1, 'Moderate': 2, 'High': 3}
FEATURES = ['avg_mvpa_min','avg_sleep_min','days_goal_met','active_days','meal_logs',
            'avg_calories','pct_healthier_choice','events_booked','programmes_enrolled',
            'healthpoints_earned','healthpoints_redeemed','vouchers_redeemed','voucher_value_sgd',
            'challenges_active','avg_challenge_progress','latest_bmi','latest_systolic',
            'screening_risk_ord']

pdf = spark.table('gold.resident_360').toPandas()
pdf['screening_risk_ord'] = pdf['screening_risk'].map(risk_map).fillna(0)
# Mix at-risk (is_disengaged == 1) and engaged residents so the demo shows both classes
sample = pd.concat([pdf[pdf['is_disengaged'] == 1].head(3),
                    pdf[pdf['is_disengaged'] == 0].head(2)])[FEATURES].astype(float).fillna(0.0)
print('Predictions (1 = at disengagement risk):', list(model.predict(sample)))

## B. Call the real-time REST endpoint (Preview)
Paste your `/score` endpoint URL from **Manage endpoints** after setting **Default version = Version 1**. The payload is a Pandas-split dataframe.

In [ ]:
import requests, json

# From the model page: set Version 1 as the default version (Manage endpoints), then copy the
# Model endpoint URL (ends with /score). No default version? Use the version URL instead:
#   .../mlmodels/<id>/endpoint/versions/1/score
ENDPOINT_URL = '<paste-endpoint-scoring-url>'

try:
    import notebookutils
    token = notebookutils.credentials.getToken('pbi')
except Exception:
    token = '<paste-a-valid-bearer-token>'

one = sample.head(1)
payload = {'formatType': 'dataframe', 'orientation': 'values',
           'inputs': one.values.tolist()}

if ENDPOINT_URL.startswith('http'):
    # First call may time out while the endpoint warms up (cold start) — just re-run this cell.
    r = requests.post(ENDPOINT_URL, headers={'Authorization': f'Bearer {token}',
                      'Content-Type': 'application/json'}, data=json.dumps(payload), timeout=120)
    print('HTTP', r.status_code)
    print('Prediction:', r.text)   # value depends on the sampled resident and your model
else:
    print('Set ENDPOINT_URL first (activate the endpoint on the model page).')